# 1. Données et qualification physique

**Question.** Que contient réellement M02 et que signifie une cible temporelle ?

Le corpus gelé précède l'entraînement. Ses observations locales et ses annotations physiques ont des rôles séparés. Nous consultons les manifestes, sans simuler ni requalifier les épisodes. Les chemins se configurent par `CGR_DATA_ROOT`, `CGR_LAB_ROOT`, `CGR_M02_ROOT` et `CGR_HISTORICAL_ROOT`. Choisir l'interpréteur Windows du projet dans VS Code.


In [ ]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from cav_recovery.laboratory.workbench import Laboratory
from cav_recovery.laboratory.storage import json_file

lab = Laboratory.from_environment()
display(HTML(lab.table_html([{'résultats': str(lab.root), 'M02': str(lab.dataset)}])))


## Effectifs et partitions

La partition est attribuée au parent géométrique, avant les fenêtres. Les trois inconnus sont retirés de la supervision et exportés séparément. Les effectifs initiaux de `splits.json` et les effectifs supervisés ne doivent pas être additionnés.


In [ ]:
partitions = lab.dataset_overview()
display(HTML(lab.table_html(partitions)))
fig = lab.plot_dataset()
plt.show()


## Cible disponible maintenant, pas étiquette rétrospective comme entrée

Le TCN estime `persistent_confirmation_available_at_end` : la confirmation physique est-elle déjà disponible à la fin de cette fenêtre ? Il n'apprend pas à prédire une intervention obligatoire et ne remplace pas une preuve K04. Les labels `retrospective_persistent_event` et `future_followup_required` restent des annotations.

Les 176 mesures potentielles couvrent 35 s à 5 Hz. Le préfixe manquant est complété à gauche avec un masque, sans inventer de mesures. Chaque fenêtre appartient à un seul CAV et à son mouvement ; les identifiants servent aux jointures, jamais aux canaux du modèle.


In [ ]:
manifest = json_file(lab.dataset / 'export-final' / 'manifest.json')
display(HTML(lab.table_html([
    {'rôle': 'Entrées autorisées', 'fichiers': ', '.join(manifest['model_input_files'])},
    {'rôle': 'Cibles privilégiées', 'fichiers': ', '.join(manifest['privileged_target_files'])},
])))
cache = json_file(lab.cache / 'complete.json')
display(HTML(lab.table_html([{'cible': cache['target'], 'oracle comme entrée': cache['oracle_as_input'],
                           'identité du corpus': cache['source_id']}])) )
print('Canaux réellement utilisés :', ', '.join(cache['features']))


## Normalisation et biais

La normalisation est ajustée sur les seules observations de train. Les positions absolues, topologies, graines, identifiants et labels ne sont pas des variables. La vitesse et la perception peuvent néanmoins refléter des biais du générateur ; une bonne mesure sur ce corpus ne suffit pas à valider un système routier réel.


In [ ]:
normalisation = json_file(lab.cache / 'normalization.json')
display(HTML(lab.table_html([{'partition ajustée': normalisation['fitted_partition'],
                            'observations': normalisation['observations'],
                            'canaux': len(normalisation['features'])}])))
limites = json_file(lab.dataset / 'production-final' / 'coverage-limits.json')
display(HTML(lab.table_html([
    {'constat': 'Négatifs avec effet établi ≥ 5 s', 'nombre': limites['negatives_with_effect_at_least_5s']},
    {'constat': 'Changements d’occupants', 'nombre': limites['changing_occupant_episodes']},
    {'constat': 'Confirmations nécessitant la succession de bloqueurs', 'nombre': limites['confirmations_requiring_successive_blockers']},
])))


**Interprétation.** Les partitions sont intactes ; le cache est une dérivation contrôlée, pas un nouveau dataset physique. L'asymétrique reste le test OOD M02. Les négatifs difficiles et certaines successions de bloqueurs restent insuffisamment représentés. Le champ historique M02 `persistence_s=5` désigne la fenêtre d'effet, et `nominal_followup_s=30` la durée qualifiante ; K04 nomme ces mêmes durées `effect_s=5` et `persistence_s=30`.
